# Lab 1: The Data Portfolio — The Economic Lens
## ECON 3916: Applied Machine Learning for Economists
### Guided Construction Lab | 42 min Core + 15 min Extension

---

**Learning Objectives:**
- Load a real-world economic dataset into a pandas DataFrame
- Distinguish cross-sectional, time series, and panel data using the same dataset
- Compute Purchasing Power Parity (PPP) exchange rates from raw prices
- Diagnose missing data patterns and classify them as MCAR, MAR, or MNAR

**Dataset:** The Economist Big Mac Index (GitHub: TheEconomist/big-mac-data)

**How this lab works:** Parts 0-2 are GUIDED: run each cell as it is, then answer the questions. Part 3 has three one-line blanks to fill in, each task followed by a check cell. The Extension is optional.

**Prerequisites:** none. Part 0 below teaches everything you are asked to *write* here: f-strings, boolean masks, double-bracket column selection, `.groupby()`.

**AI in the manual parts:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion while you work on those parts. AI-assisted coding is allowed only where this lab says so.

---

> ### What you write, and what you run
>
> **You write:**
> - **Part 3, Tasks 5b and 6:** three blanks (`____`), one line each: `incomplete`, `df_russia` and `last_date`. A CHECK cell after each task tells you whether it is right.
> - The answers to the interpretation questions in Parts 1 and 2, and four rows of the missing-data table in Part 3 plus its key question, in text cells.
> - **Extension (optional):** change the four country names in one list, run the cell again, and write one sentence about the chart.
> - A README for your GitHub repository, drafted with AI from the prompt near the end. It is documentation, not code.
>
> **You run and read:** everything else. Part 0, Parts 1 and 2, the rest of Part 3 and the Extension are working code to learn from. Read each cell before you run it, and make sure you can say what each line does.
>
> **Reading this after the due date?** Use Lab 1, with its solution, as a worked example while you write Problem Set 1. PS1 asks you to write code from a blank cell, and the patterns you will reach for most (f-strings, boolean masks, `.groupby()`) are all here in working code.

## Setup

In [ ]:
# GUIDED — run as-is
# Step 1: load the libraries and the Big Mac Index

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# The Economist publishes the Big Mac Index as a CSV file on GitHub
url = "https://raw.githubusercontent.com/TheEconomist/big-mac-data/master/output-data/big-mac-full-index.csv"
df = pd.read_csv(url, parse_dates=["date"])   # read the date column as dates, not text

# A new Colab session has no figures/ folder, and savefig will not create one
Path("figures").mkdir(exist_ok=True)

print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print("\nFirst 5 rows:")
df.head()

**Expected output:** a shape of (2056, 19): 2,056 rows and 19 columns. The `date` column's type is `datetime64`, not `str`; that is what `parse_dates` did. Below it, the first five rows of the table. The Economist adds an edition each January and July, so a later run may show a few more rows.

<!-- 3916-onramp -->
## Part 0: The Floor (GUIDED — 12 min)

**Read this before anything else, and run every cell.** This course assumes no
prior programming, so Part 0 of Labs 1 through 5 teaches the slice of Python
that lab needs, next to the data that needs it.

**This lab's slice:** the notebook itself, f-strings, the DataFrame, the
boolean mask, and `.groupby()`. Everything Part 3 asks you to *write* is
shown here first.

### The notebook

You are in a **notebook**: a document of **cells**, each holding prose or code.
Run a code cell with **Shift+Enter**; the output appears underneath.

- **Kernel** — the Python process behind the notebook. It remembers everything
  you have defined, and all cells share one, so **order matters**: skip a cell
  and the ones after it may fail on a name that was never created.
- **Restart** — *Runtime → Restart session* empties it. When a notebook behaves
  impossibly, restart and run from the top. This fixes more than anything else.

When a cell errors, read the **last** line of the message first. That is the
actual complaint; the wall above it is only the route Python took to get there.

### Step 0a: variables and the f-string

A **variable** is a name bound to a value with a single `=`. Read `x = 5` as "let x refer to 5", not "x equals 5". Every value has a **type**: `str` for text in quotes, `float` for a number with a decimal point, `int` for a whole number.

An **f-string** is a string with `f` before the opening quote. Anything inside `{ }` is run as Python, and its value is dropped into the text. It is the construct you will use most in this course.

The part after a colon inside the braces is a **format spec**. It changes how a number is *displayed*, never the number itself:

| Spec | Meaning | Example |
|---|---|---|
| `:.2f` | fixed point, 2 decimals | 7.1 → 7.10 |
| `:.1%` | percent, 1 decimal | 0.042 → 4.2% |
| `:,.0f` | thousands separator, no decimals | 2056 → 2,056 |
| `:<15` | left-align in 15 characters | for lining up tables |

`:.1%` multiplies by 100 for you, so it suits a **fraction** such as 0.4176. Part 2's `valuation_pct` column is already multiplied by 100 (41.76): format that one with `:.1f` and type the `%` yourself, or `:.1%` prints 4175.6%.

The last two lines are a **`for` loop**: it runs the indented line once for each item in the list. Each item here is a pair, and `for name, pct in ...` unpacks the pair into two names.

In [ ]:
# GUIDED — run as-is
# Step 0a: variables, types, and the f-string

country = "Switzerland"
big_mac = 7.10            # the Swiss price, in francs, not dollars
n_periods = 45
print(type(country), type(big_mac), type(n_periods))

valuation = 0.4176        # a fraction, which is what :.1% expects
print(f"{country}: CHF {big_mac:.2f}, valued {valuation:.1%} against the US")

# The most common f-string bug: multiplying by 100 before using :.1%
print(f"WRONG: {valuation * 100:.1%}   <- 0.4176 * 100 = 41.76, shown as 4176.0%")
print(f"RIGHT: {valuation:.1%}")

# :<15 pads the name to 15 characters; :>8.1% right-aligns the percent in 8
print()
for name, pct in [("Switzerland", 0.4176), ("Norway", 0.1894), ("Taiwan", -0.5990)]:
    print(f"  {name:<15}{pct:>8.1%}")

### Step 0b: the DataFrame, one bracket or two, and the boolean mask

`df`, loaded in Setup, is a **DataFrame**: a table with named columns and ordered rows. The **index** is the row labels down the left; it is not a column. A column's **dtype** is its type, and many pandas errors come from a dtype that is not what you assumed.

- **One pair of brackets** with one name, `df["name"]`, gives a **Series**: a single column.
- **Two pairs**, `df[["name", "local_price"]]`, give a **DataFrame**, because the inner brackets are a *list* of names.

A **boolean mask** is the pattern you will use most in this course. A comparison on a column gives one True or False per row, and putting that inside `df[ ... ]` keeps only the rows where it is True. Python counts True as 1 and False as 0, so `.sum()` of a mask counts the Trues.

To combine conditions, use `&` (and), `|` (or) and `~` (not), never the words `and`, `or`, `not`, which raise *"ValueError: The truth value of a Series is ambiguous"*. Wrap each condition in its own parentheses:

```python
df[(df["a"] > 1) & (df["b"] < 2)]     # correct
df[ df["a"] > 1  &  df["b"] < 2 ]     # wrong: & is applied before >
```

In [ ]:
# GUIDED — run as-is
# Step 0b: the DataFrame, one bracket vs two, and boolean masks

# .nunique() counts the distinct values in a column
print(f"Shape (rows, columns): {df.shape}")
print(f"Countries: {df['name'].nunique()}   Periods: {df['date'].nunique()}")

print()
print("df['name']                  is", type(df["name"]))
print("df[['name']]                is", type(df[["name"]]))
print("df[['name','local_price']]  is", type(df[["name", "local_price"]]))

# One True/False per row: True where the row is from July 2024
mask = df["date"] == "2024-07-01"
print(f"\nThe mask is {len(mask)} True/False values; {mask.sum()} are True.")
print(f"Countries in that cross-section: {len(df[mask])}")

dear = df[(df["date"] == "2024-07-01") & (df["dollar_price"] > 6)]
print(f"\nBig Macs over $6 in July 2024: {len(dear)}")
print(dear[["name", "dollar_price"]].to_string(index=False))   # index=False hides the row labels

# Filtering to one country is the same move, and you will write it in Part 3
russia = df[df["name"] == "Russia"]
print(f"\nRussia appears {len(russia)} times; last on {russia['date'].max().date()}.")   # .date() drops the time of day

### Step 0c: `.groupby()`, and masking what it returns

`.groupby(col)` splits the table into one group per distinct value of `col`, applies a calculation to each group, and combines the answers: split, apply, combine.

Read the last lines of the cell twice: they are the shape Part 3 asks you to write. Step 0b masked a DataFrame using one of its columns, `df[df["name"] == "Russia"]`. Here the thing you compare and the thing you filter are the **same Series**. `country_counts == max_periods` is one True/False per *country*, and `country_counts[ ... ]` keeps the countries where it is True. The rule is the same as before; only the source of the mask has changed. `len()` of the result counts countries, not rows of `df`.

In [ ]:
# GUIDED — run as-is
# Step 0c: .groupby(), and masking what it returns

# .dt.year takes the year from each date, so this is one average per year
yearly = df.groupby(df["date"].dt.year)["dollar_price"].mean()
print("Mean dollar price by year (first five):")
print(yearly.head().round(2))

# One count per country: a Series whose index is the country names
max_periods = df["date"].nunique()
country_counts = df.groupby("name")["date"].count()
print("\ncountry_counts is", type(country_counts))
print(f"It holds {len(country_counts)} countries. First three:")
print(country_counts.head(3))

# The mask and the Series it filters are the same Series
complete = country_counts[country_counts == max_periods]
print(f"\nCountries present in all {max_periods} periods: {len(complete)} of {len(country_counts)}")

## Part 1: Data Structure Exploration (GUIDED — 10 min)

The Big Mac Index is a **panel dataset** — many countries observed over many time periods.
From this single dataset, we can extract all three data structures.

In [ ]:
# GUIDED — run as-is
# Step 2: three data structures from one dataset

n_countries = df["name"].nunique()
n_periods = df["date"].nunique()
print(f"Countries: {n_countries}, Time periods: {n_periods}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print()

# Cross-sectional: every country at one date
df_cross = df[df["date"] == "2024-07-01"][["name", "local_price", "dollar_price"]]
print(f"Cross-sectional (July 2024): {len(df_cross)} countries, 1 time period")
print()

# Time series: one country at every date
df_ts = df[df["name"] == "Switzerland"][["date", "local_price", "dollar_price"]]
print(f"Time series (Switzerland): 1 country, {len(df_ts)} time periods")
print()

# Panel: every country at every date, which is the whole file
print(f"Panel (full dataset): {n_countries} countries, {n_periods} periods, {len(df)} observations")

**Expected output:** 57 countries over 45 periods, from 2000-04-01 to 2026-07-01; 54 countries in the July 2024 cross-section; 45 periods for Switzerland; 2,056 rows in the full panel. The panel is **unbalanced**: not every country appears in every period. Question 1 asks why.

### Interpretation Questions (answer in the cell below)

1. Why is the panel "unbalanced"? What does that mean practically?
2. If you wanted to ask "Which country has the most expensive Big Mac right now?", which structure would you use?
3. If you wanted to ask "Has Switzerland's Big Mac price been rising over time?", which structure would you use?

*Your answers here:*

1. 
2. 
3. 

## Part 2: Computing PPP — The Big Mac Index (GUIDED — 10 min)

Purchasing Power Parity says exchange rates should equalize prices across countries.
The Big Mac Index tests this with one product.

Step 3 computes two numbers per country, both against the US price:

- **Implied PPP** = local price ÷ US dollar price. It is the exchange rate that would make a Big Mac cost the same in both countries.
- **Valuation (%)** = (implied PPP − actual exchange rate) ÷ actual exchange rate × 100. Above zero, the currency is overvalued: a Big Mac there costs more, in dollars, than in the US. The actual exchange rate is the `dollar_ex` column, in local currency per dollar.

In [ ]:
# GUIDED — run as-is
# Step 3: implied PPP and valuation, July 2024

# .copy() makes df_2024 a table of its own, so the columns added below do not touch df
df_2024 = df[df["date"] == "2024-07-01"].copy()

# The US price is the benchmark; .iloc[0] takes the first (and only) value
us_row = df_2024[df_2024["iso_a3"] == "USA"]
us_price = us_row["dollar_price"].iloc[0]
print(f"US Big Mac price: ${us_price:.2f}")

df_2024["implied_ppp"] = df_2024["local_price"] / us_price
df_2024["valuation_pct"] = (
    (df_2024["implied_ppp"] - df_2024["dollar_ex"]) / df_2024["dollar_ex"] * 100
)

cols = ["name", "local_price", "dollar_price", "implied_ppp", "dollar_ex", "valuation_pct"]

# nlargest(5, col) keeps the 5 rows with the largest values of col
print("\n--- Most OVERVALUED (expensive) ---")
print(df_2024.nlargest(5, "valuation_pct")[cols].to_string(index=False))

print("\n--- Most UNDERVALUED (cheap) ---")
print(df_2024.nsmallest(5, "valuation_pct")[cols].to_string(index=False))

**Expected output:** a US price of \$5.69. Switzerland heads the overvalued list at +41.8%, and Taiwan heads the undervalued list at −59.9%. To read a row, compare two columns: Switzerland's `implied_ppp` of 1.25 francs per dollar is well above its actual rate, `dollar_ex`, of 0.88, so a Swiss Big Mac is dear in dollars.

In [ ]:
# GUIDED — run as-is
# Step 4: every country's valuation on one chart

df_sorted = df_2024.sort_values("valuation_pct")
cheaper = df_sorted[df_sorted["valuation_pct"] < 0]
dearer = df_sorted[df_sorted["valuation_pct"] >= 0]

# fig is the whole image and ax the chart inside it; each ax. line draws or labels that chart
fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(cheaper["name"], cheaper["valuation_pct"], color="indianred", label="Undervalued")
ax.barh(dearer["name"], dearer["valuation_pct"], color="steelblue", label="Overvalued")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Valuation vs. US (%)")
ax.set_title("Most currencies were undervalued against the dollar in July 2024")
ax.tick_params(axis="y", labelsize=7)   # small type, so 54 country names do not overlap
ax.legend()
plt.tight_layout()
plt.savefig("figures/ch01_bigmac_valuation.png", dpi=150)
plt.show()

print(f"\nMedian valuation: {df_2024['valuation_pct'].median():.1f}%")
print(f"Countries more expensive than US: {(df_2024['valuation_pct'] > 0).sum()} of {len(df_2024)}")

**Expected output:** one bar per country, from Taiwan at the bottom (most undervalued) to Switzerland at the top. Below the chart: a median valuation of −20.7%, and 6 of 54 countries more expensive than the US.

### Interpretation Questions

1. Why are most countries UNDERVALUED relative to the US? What does this tell you about the US price level?
2. Switzerland has been the most overvalued currency in every period since January 2015. What non-tradeable inputs explain this? (Hint: rent, wages, taxes)
3. **Critical question:** If a hedge fund used this chart to make currency bets, would you expect them to make money? Why or why not?

*Your answers here:*

1. 
2. 
3. 

## Part 3: Data Quality & Missing Data (YOUR TASK — 10 min)

Now you investigate the dataset's missing data: which countries have gaps, and why?

In [ ]:
# GUIDED — run as-is
# Step 5a: missing values per column

# .isnull() marks each empty cell True, and .sum() counts them column by column
print("Missing values per column:")
print(df.isnull().sum())

**Expected output:** zero in every price column. The only nulls are 254 in each of the seven GDP-adjusted columns, `GDP_bigmac` to `CNY_adjusted`. A country that is absent from an edition has no row at all for that date, so it cannot show up as a null here. Task 5b counts those gaps a different way.

In [ ]:
# YOUR TASK — fill in the blank (____)
# Task 5b: countries with incomplete panels

max_periods = df["date"].nunique()
country_counts = df.groupby("name")["date"].count()   # periods per country, as in Step 0c

# Keep the countries that appear in fewer than max_periods periods
incomplete = ____

print(f"Countries with incomplete panels ({len(incomplete)} of {df['name'].nunique()}):")
print(incomplete.sort_values())

In [ ]:
# CHECK — run as-is after Task 5b

assert isinstance(incomplete, pd.Series), (
    f"incomplete is a {type(incomplete).__name__}; it should be a Series with one "
    "entry per country -- filter country_counts, not df.")
assert incomplete.dtype != bool, (
    "incomplete holds True/False values. That is the mask itself; put it inside "
    "country_counts[ ... ] to keep the countries where it is True.")
assert (incomplete < max_periods).all(), (
    "incomplete contains a country that appears in every period.")
n_complete = int((country_counts == max_periods).sum())
assert len(incomplete) + n_complete == len(country_counts), (
    f"incomplete has {len(incomplete)} countries and {n_complete} are complete, but "
    f"there are {len(country_counts)} countries: some incomplete ones are missing.")
print(f"Task 5b check passed: {len(incomplete)} incomplete + {n_complete} complete "
      f"= {len(country_counts)} countries.")

In [ ]:
# YOUR TASK — fill in the two blanks (____)
# Task 6: investigate Russia's disappearance

# Filter df, the whole panel: Russia is not in df_2024, so that would give zero rows
df_russia = ____

print("Russia's last observations:")
print(df_russia[["date", "local_price", "dollar_price"]].tail())   # .tail() shows the last 5 rows

# The last date on which Russia appears
last_date = ____
print(f"\nRussia's last appearance: {last_date}")
print(f"Russia's total observations: {len(df_russia)} of {max_periods} possible")

In [ ]:
# CHECK — run as-is after Task 6

assert isinstance(df_russia, pd.DataFrame) and len(df_russia) > 0, (
    "df_russia is empty or not a DataFrame -- filter df, the whole panel.")
others = sorted(set(df_russia["name"]) - {"Russia"})
assert not others, f"df_russia also contains rows for {others[:3]}."
assert len(df_russia) == country_counts["Russia"], (
    f"df_russia has {len(df_russia)} rows; Russia appears {country_counts['Russia']} "
    "times in the panel.")
last = pd.Timestamp(last_date)
assert last in set(df_russia["date"]) and (df_russia["date"] <= last).all(), (
    f"{last_date} is not the latest date in df_russia.")
print(f"Task 6 check passed: {len(df_russia)} observations, last on {last.date()}.")

### Missing Data Classification

32 countries have incomplete panels, but you do **not** classify all 32 — most of
them repeat the same story. These five rows are five *different* stories. Russia is
done for you; fill in the other four.

Print a country's dates before you classify it:

```python
df[df["name"] == "Venezuela"]["date"].dt.strftime("%Y-%m").tolist()
```

`.dt.strftime("%Y-%m")` writes each date as year and month, and `.tolist()` prints the
result as a plain list.

Then ask: did it **leave**, **join late**, or have **gaps in the middle** of a run
that is still going? Those are different mechanisms, and one country can have more
than one kind of gap.

| Country | What the dates show | Likely mechanism | Reasoning |
|---------|---------------------|------------------|-----------|
| Russia | last seen 2022-01; 9 editions published since | MNAR | McDonald's left after the invasion — no Big Mac to price |
| Venezuela | | | *Hint: which years are missing?* |
| Qatar | | | *Hint: who else first appears on the same date?* |
| Ukraine | | | *Hint: look at 2022 — then check Denmark* |
| UAE | | | *Hint: compare with `United Arab Emirates`* |

**Key question:** If you drop all countries with incomplete panels and compute the
"average PPP deviation," is your result biased? In which direction?

*Your answers here:*



---

## Extension (Optional — 15 min): Time Series Visualization

Track how PPP valuation changes over time for selected countries.

Part 2 used one US price, July 2024's. Across all 45 periods the US price changes, so each country has to be compared with the US price **of the same date**. Step 7 does that with a **merge**: it takes the US row for each date and attaches its price to every row with that date. Step 8 plots four countries. Run both as written first; the task is under the chart.

In [ ]:
# EXTENSION — run as-is
# Step 7: valuation at every date, not just July 2024

# One row per date holding the US price, renamed so it does not clash with dollar_price
us_prices = df[df["iso_a3"] == "USA"][["date", "dollar_price"]]
us_prices = us_prices.rename(columns={"dollar_price": "us_price"})   # {old name: new name}

# Attach each date's US price to every row with that date; every date has one
df_full = df.merge(us_prices, on="date", how="inner")
df_full["implied_ppp"] = df_full["local_price"] / df_full["us_price"]
df_full["valuation_pct"] = (
    (df_full["implied_ppp"] - df_full["dollar_ex"]) / df_full["dollar_ex"] * 100
)

In [ ]:
# EXTENSION — run as-is first, then change the four countries and run it again
# Step 8: valuation over time for four countries

countries = ["Switzerland", "Japan", "Brazil", "China"]

fig, ax = plt.subplots(figsize=(12, 6))
for country in countries:
    subset = df_full[df_full["name"] == country]
    ax.plot(subset["date"], subset["valuation_pct"], label=country, linewidth=2)

ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_ylabel("Valuation vs. US (%)")
ax.set_title("Switzerland was dearer than the US in every edition, China cheaper in every one")
ax.legend()
plt.tight_layout()
plt.savefig("figures/ch01_bigmac_timeseries.png", dpi=150)
plt.show()

**Your task (optional):** replace the four names in `countries` with four countries you care about, and run the cell again. Then write one sentence below: which of your four has the market repriced most since 2010? If you change the countries, change the title too, so it still states what your chart shows. To list every valid name, run `sorted(df["name"].unique())` in a new cell.

*Your sentence here:*

---
## Digital Portfolio: Your README

### Draft your README with AI
Copy the prompt below into Claude or ChatGPT, and replace each bracket with the number from your own output first. **Do NOT ask the AI to write Python code, only documentation.**

```
I need help writing a project README for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Loaded The Economist's Big Mac Index from GitHub: [N] countries
  over [N] periods, [N] of them in the July 2024 cross-section
* Computed Purchasing Power Parity exchange rates and valuation percentages
* Identified data structure types (cross-sectional, time series, panel)
* Diagnosed missing data patterns and classified Russia's exit as MNAR
* Made a bar chart of July 2024 valuations and a line chart of
  valuation over time
* Key finding: in July 2024 Switzerland was [YOUR VALUE]% overvalued
  against the US, and only [N] of [N] countries were more expensive
  than the US

**Please write a README.md entry including:**
1. Project Title: The Data Portfolio — Big Mac Index Analysis
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust',
'comprehensive' or 'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ3916-lab01-data-portfolio`

**First time?** Read the **GitHub Setup** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ3916-lab01-data-portfolio`, tick **Add a README file**, and create it.
2. **Run the whole notebook top to bottom** (**Runtime → Run all**) and check your
   output is showing. The rendered output — charts included — is what is graded.
3. In Colab: **File → Save a copy in GitHub**.
   - **First time only:** a GitHub window asks you to authorize Colab. Sign in, then
     authorize. **If your repo is Private, tick the box granting access to private
     repositories first** — otherwise Colab won't list your repo.
   - Choose `econ3916-lab01-data-portfolio`, branch `main`, commit message
     `Lab 01 submission`, tick **Include a link to Colab**, and click **OK**.
4. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
5. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your **repository** URL in the **Comments**
   box — `https://github.com/yourname/econ3916-lab01-data-portfolio`, not the
   link to the notebook file. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install. Stuck? The GitHub Setup page has a
Troubleshooting section.
